In [23]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import heapq
import seaborn as sns

from environment.environment import GraphWorldMFG_MultiGroup
from trainer.amid_trainer_graph import GraphEdgeMFG_Trainer
from solver.solver import solve_multigroup, GraphMFG_OMD_EdgeSolver_MultiGroup
from visualization.visualizationh import plot_heatmap, compute_exploitability_multigroup, plot_losses, plot_losses_line

In [24]:
import os
from pathlib import Path
import yaml
import numpy as np
import torch

# 1. Configuration Loader
def load_config(config_path="config.yaml"):
    """Load configuration safely from a YAML file relative to working directory."""
    notebook_dir = Path(os.getcwd())
    config_file = notebook_dir / config_path
    
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    return config

# 2. Graph Environment Factory Pattern
def create_graph_mfg_from_config(config):
    """Factory function initializing the Graph MFG environment directly from your config."""
    device = config.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Device: {device}")

    trainer_cfg = config["trainer"]
    
    # Extract Graph Configuration
    graph_cfg = config["graph"]
    num_nodes = graph_cfg["num_nodes"]
    
    # CRITICAL: Extract the list of lists matrix and convert to a PyTorch Tensor
    raw_matrix = graph_cfg["adjacency_matrix"]
    adjacency_matrix = torch.tensor(raw_matrix, dtype=torch.float32, device=device)
    
    # Process Group Data (Sinks and Sources are flat integers here, not grid tuples)
    groups = []
    for g in config["groups"]:
        groups.append({
            "source": int(g["source"]),
            "sink": int(g["sink"]),
            "mass": float(g["mass"])
        })
        
    # Instantiate the Graph World Environment
    # (Matches your 'GraphWorldMFG_MultiGroup' class structure)
    env = GraphWorldMFG_MultiGroup(
        num_nodes=num_nodes,
        groups=groups,
        adjacency_matrix=adjacency_matrix,
        device=device
    )
    
    # Create solvers for each group
    solver_cfg = config["solver"]
    solvers = [
        GraphMFG_OMD_EdgeSolver_MultiGroup(
            env=env,
            group_idx=k,
            eta=solver_cfg["eta"],
            tau=solver_cfg["tau"],
            T=solver_cfg["T"],
            alpha=solver_cfg["alpha"],
            H=solver_cfg["H"]
        )
        for k in range(env.K)
    ]

    trainer = GraphEdgeMFG_Trainer(env, solvers, leader_lr=trainer_cfg["leader_lr"])
    
    return env, solvers, trainer, config

In [25]:
config = load_config("config_graph.yaml")

env, solvers, trainer, config = create_graph_mfg_from_config(config)

# Training loop
losses = []

Target Device: cpu
1


In [26]:
solvers[0].base_thetas

tensor([[[-0., -0., -0., -0., -0., -0.],
         [-0., -0., -0., -0., -0., -0.],
         [-0., -0., -0., -0., -0., -0.],
         [-0., -0., -0., 0., -0., -0.],
         [-0., -0., -0., -0., -0., -0.],
         [-0., -0., -0., -0., -0., -0.]]])

In [27]:
#trainer.base_thetas += -0.35
#trainer.base_thetas[0,5,5] = 6  # No congestion cost at the sink for group 0
trainer.base_thetas

tensor([[[-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000],
         [-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000],
         [-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000],
         [-0.0000, -0.0000, -0.0000,  6.0000, -0.0000, -0.0000],
         [-0.0000, -0.0000, -1.1250, -0.0000, -0.0000, -0.0000],
         [-0.0000, -0.0000, -0.0000, -1.1250, -0.0000, -0.0000]]])

In [28]:
policies, flows, final_flows = solve_multigroup(solvers, trainer.base_thetas)

Running multi-group OMD for T=200 steps...
Step 200/200 completed. Final policies and flows computed.
Multi-group OMD completed.


In [29]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [7.9713e-34, 9.8103e-01, 0.0000e+00, 0.0000e+00, 1.8967e-02, 0.0000e+00],
        [0.0000e+00, 7.9713e-34, 9.8083e-01, 0.0000e+00, 2.5167e-42, 1.9173e-02],
        [0.0000e+00, 0.0000e+00, 7.9713e-34, 1.0000e+00, 0.0000e+00, 1.5793e-42],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00]],
       grad_fn=<SumBackward1>)

In [30]:
def compute_social_loss(self, flows, policies, theta_leader):
    """Calculates systemic social reward using link-level traffic contraction."""
    total_social_reward = 0.0

    # Compute total edge traffic across all groups for each time step.
    edge_flows = [flow.unsqueeze(-1) * policy for flow, policy in zip(flows, policies)]
    E_total_edges = torch.stack(edge_flows).sum(dim=0)  # (H, N, N)

    for k in range(self.env.K):
        flow_k = flows[k]        # (H, N)
        policy_k = policies[k]  # (H, N, N)

        for h in range(self.solvers[k].H):
            # 1. Compute this group's specific link utilization matrix
            E_kh = flow_k[h].unsqueeze(-1) * policy_k[h]  # Shape: (N, N)

            # 2. Use the true total edge utilization across all groups
            E_total_h = E_total_edges[h]
            
            E_total_final = torch.zeros_like(E_total_h)
            E_total_final[0,1] = E_total_h[0,1]
            E_total_final[2,3] = E_total_h[2,3]
            
            # 3. Aggregate Edge Reward Matrix
            reward_matrix = -self.solvers[k].alpha * E_total_final + theta_leader[0]
            reward_matrix[self.env.groups[k]["sink"], self.env.groups[k]["sink"]] = 0.0  # No congestion cost at the sink

            # 4. Perform matrix dot product contraction
            step_reward = torch.sum(E_kh * reward_matrix)
            total_social_reward += step_reward

            print(step_reward)

    return -total_social_reward

In [31]:
cost = compute_social_loss(trainer, flows, policies, trainer.base_thetas)  
print("Computed Social Loss:", cost.item())

tensor(-0.9624, grad_fn=<SumBackward0>)
tensor(-0.0213, grad_fn=<SumBackward0>)
tensor(-0.9836, grad_fn=<SumBackward0>)
tensor(-1.7768e-42, grad_fn=<SumBackward0>)
tensor(0., grad_fn=<SumBackward0>)
tensor(0., grad_fn=<SumBackward0>)
tensor(0., grad_fn=<SumBackward0>)
Computed Social Loss: 1.9673552513122559


In [32]:
print("\nEpoch | Loss")
print("-" * 25)

policies, flows, final_flows = solve_multigroup(solvers, trainer.base_thetas, number_epochs= 1)

for epoch in range(config["trainer"]["num_epochs"]):
    loss, final_flows, flows, policies = trainer.train_step(final_flows, flows, policies)
    losses.append(loss)
    if epoch % 1 == 0:
        print(f"{epoch:5d} | {loss:.6f}")


Epoch | Loss
-------------------------
Running multi-group OMD for T=1 steps...
Step 1/1 completed. Final policies and flows computed.
Multi-group OMD completed.
base_thetas in train_step torch.Size([1, 6, 6])
Running multi-group OMD for T=200 steps...
Step 200/200 completed. Final policies and flows computed.
Multi-group OMD completed.
    0 | 3.557242
base_thetas in train_step torch.Size([1, 6, 6])
Running multi-group OMD for T=200 steps...
Step 200/200 completed. Final policies and flows computed.
Multi-group OMD completed.
    1 | 3.461299
base_thetas in train_step torch.Size([1, 6, 6])
Running multi-group OMD for T=200 steps...
Step 200/200 completed. Final policies and flows computed.
Multi-group OMD completed.
    2 | 3.325158
base_thetas in train_step torch.Size([1, 6, 6])
Running multi-group OMD for T=200 steps...
Step 200/200 completed. Final policies and flows computed.
Multi-group OMD completed.
    3 | 3.135655
base_thetas in train_step torch.Size([1, 6, 6])
Running multi

KeyboardInterrupt: 

In [ ]:
def compute_graph_exploitability(env, solver, current_policy, current_flow):
    """
    Computes the exact unilateral exploitability for a single group on the graph.
    """
    import torch
    
    # 1. Re-calculate the total edge congestion under the current converged scenario
    # (Assuming single-group for simplicity; sum across groups if multi-group)
    E_total_edges = solver.compute_edge_traffic(current_policy, current_flow)
    
    # 2. Compute the ACTUAL Q-values under the current traffic, but with NO entropy (tau=0)
    # This represents the true objective environmental cost of every path.
    q_list = []
    V_next_BR = torch.zeros(env.N, device=env.device)  # Value function for Best Response
    V_next_curr = torch.zeros(env.N, device=env.device) # Value function for Current Policy
    
    # Run the Bellman backward pass from horizon H-1 down to 0
    for h in reversed(range(solver.H)):
        # Construct raw edge rewards for this step
        reward_matrix = -solver.alpha * E_total_edges[h] + solver.base_thetas[0] # assuming group 0
        reward_matrix[env.groups[0]["sink"], env.groups[0]["sink"]] = 0.0  # Zero out sink penalty
        
        # Immediate Q-values for this time-step
        # Q(s, s') = Reward(s, s') + Value_of_next_state(s')
        current_q_BR = reward_matrix + V_next_BR.unsqueeze(0)
        current_q_curr = reward_matrix + V_next_curr.unsqueeze(0)
        
        # --- BEST RESPONSE VALUE (Hard Max) ---
        # The absolute best choice an exploiting agent could make
        # Mask out unreachable nodes (where M == 0)
        masked_q_BR = current_q_BR.clone()
        masked_q_BR[env.M == 0] = float('-inf')
        V_next_BR = torch.max(masked_q_BR, dim=-1).values
        # Re-zero sink
        V_next_BR[env.groups[0]["sink"]] = 0.0
        
        # --- CURRENT POLICY VALUE (Expected Value) ---
        # What your agents are actually experiencing under current_policy
        pi_h = current_policy[h]
        expected_q = torch.diagonal(torch.matmul(pi_h, current_q_curr.t()))
        V_next_curr = expected_q # No entropy added here; we want raw reward comparison
        V_next_curr[env.groups[0]["sink"]] = 0.0

    # 3. Read the values at the source node (Node 0) at time step 0
    source_node = env.groups[0]["source"]
    
    val_best_response = V_next_BR[source_node].item()
    val_current_policy = V_next_curr[source_node].item()
    
    # Exploitability is the gap (expressed as a positive loss/cost difference)
    # Since rewards are negative losses, (Current Policy Loss) - (Best Response Loss)
    exploitability = val_current_policy - val_best_response
    
    print(f"--- Exploitability Analysis ---")
    print(f"Expected Return of Best Exploiting Path: {val_best_response:.4f}")
    print(f"Expected Return of Current 50-50 Policy  : {val_current_policy:.4f}")
    print(f"Unilateral Exploitability Gap          : {exploitability:.4f}")
    
    return exploitability

compute_graph_exploitability(env, trainer.solvers[0], policies[0], flows[0])

--- Exploitability Analysis ---
Expected Return of Best Exploiting Path: 0.0000
Expected Return of Current 50-50 Policy  : -3.0000
Unilateral Exploitability Gap          : -3.0000


-3.0

In [ ]:
inp = trainer._prepare_input()
theta_leader = trainer.leader_nets(inp)  # Shape: (K, N, N)

print("theta_leader in train_step", theta_leader.shape)
print("base_thetas in train_step", trainer.base_thetas.shape)

# Broadcast alignment with the base static matrices
theta_final = theta_leader + trainer.base_thetas  # Shape: (K, N, N)

# solve_multigroup structure stays identical; handles N x N strategy spaces naturally
policies, flows, final_flows = solve_multigroup(trainer.solvers, theta_final)

TypeError: GraphEdgeMFG_Trainer._prepare_input() missing 3 required positional arguments: 'final_flows', 'flows_groups', and 'policies'

In [ ]:
trainer.base_thetas[0]

In [ ]:
theta_leader

In [ ]:
theta_leader + trainer.base_thetas

In [ ]:
a =theta_leader[0].cpu().detach().numpy()
a = a * env.M.cpu().detach().numpy()
plot_heatmap(a, title="Theta Leader (Group 0)", xlabel="Column", ylabel="Row")


In [ ]:
env.dist_maps

In [ ]:
final_flows

In [ ]:
plot_losses_line({"Total leader loss": losses})

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

# Define a 4x4 adjacency matrix (Unweighted)
# Node 0 connects to 1 and 2; Node 1 connects to 2, etc.
adj_matrix = env.M.cpu().detach().numpy()  - np.eye(env.N) # Convert PyTorch tensor to NumPy array

# 1. Create a graph object from the adjacency matrix
G = nx.from_numpy_array(adj_matrix)

# 2. Choose a layout for the nodes (spring layout looks nice and organic)
pos = nx.spring_layout(G)

# 3. Draw the graph
plt.figure(figsize=(6, 6))
nx.draw(
    G,
    pos,
    with_labels=True,
    node_color="skyblue",
    node_size=800,
    edge_color="gray",
    font_size=15,
    font_weight="bold",
)
plt.title("Graph Visualization from Unweighted Adjacency Matrix")
plt.show()